# Setup

In [1]:
import numpy as np
import pandas as pd

In [2]:
DIR = '/kaggle/input/playground-series-s5e5'

train = pd.read_csv(f'{DIR}/train.csv')
test = pd.read_csv(f'{DIR}/test.csv')

# Overview

In [3]:
train.shape, test.shape

((750000, 9), (250000, 8))

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          750000 non-null  int64  
 1   Sex         750000 non-null  object 
 2   Age         750000 non-null  int64  
 3   Height      750000 non-null  float64
 4   Weight      750000 non-null  float64
 5   Duration    750000 non-null  float64
 6   Heart_Rate  750000 non-null  float64
 7   Body_Temp   750000 non-null  float64
 8   Calories    750000 non-null  float64
dtypes: float64(6), int64(2), object(1)
memory usage: 51.5+ MB


In [5]:
train.head()

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,4,female,38,166.0,61.0,25.0,102.0,40.6,146.0


In [6]:
TARGET = 'Calories'
train[TARGET].describe()

count    750000.000000
mean         88.282781
std          62.395349
min           1.000000
25%          34.000000
50%          77.000000
75%         136.000000
max         314.000000
Name: Calories, dtype: float64

In [7]:
train = train.drop('id', axis=1)
test_idx = test.pop('id')  # for submission file

In [8]:
FEATURES = list(test.columns)
CAT_COLS = ['Sex']
NUM_COLS = [f for f in FEATURES if f not in CAT_COLS]

# Removing duplicates

Complete duplicates -> same features + same target

In [9]:
train.loc[train.duplicated(keep=False)].sort_values(FEATURES)

,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
106013,female,20,154.0,59.0,4.0,86.0,38.9,16.0
107865,female,20,154.0,59.0,4.0,86.0,38.9,16.0
199295,female,20,157.0,58.0,2.0,79.0,37.9,5.0
460784,female,20,157.0,58.0,2.0,79.0,37.9,5.0
172526,female,20,158.0,54.0,10.0,97.0,39.9,52.0
...,...,...,...,...,...,...,...,...
668792,male,79,182.0,86.0,26.0,110.0,40.7,240.0
324720,male,79,184.0,86.0,24.0,104.0,40.8,196.0
453176,male,79,184.0,86.0,24.0,104.0,40.8,196.0
189424,male,79,188.0,93.0,29.0,109.0,40.9,264.0


In [10]:
train = train.drop_duplicates(keep='first', ignore_index=True)

Pseudo duplicates -> same features + different targets

In [11]:
train.loc[train.duplicated(subset=FEATURES, keep=False)].sort_values(list(train.columns))

,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
575192,female,20,149.0,54.0,27.0,104.0,40.7,156.0
75553,female,20,149.0,54.0,27.0,104.0,40.7,162.0
35099,female,20,151.0,54.0,19.0,96.0,40.6,94.0
546025,female,20,151.0,54.0,19.0,96.0,40.6,99.0
396033,female,20,152.0,54.0,14.0,96.0,40.2,67.0
...,...,...,...,...,...,...,...,...
594891,male,76,189.0,93.0,29.0,112.0,40.8,259.0
595662,male,78,174.0,77.0,24.0,107.0,40.7,202.0
738755,male,78,174.0,77.0,24.0,107.0,40.7,204.0
728441,male,79,182.0,83.0,26.0,110.0,40.8,240.0


In [12]:
train[TARGET] = train.groupby(FEATURES)[TARGET].transform('mean')

In [13]:
train.loc[train.duplicated(subset=FEATURES, keep=False)].sort_values(list(train.columns))

,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
75553,female,20,149.0,54.0,27.0,104.0,40.7,159.0
575192,female,20,149.0,54.0,27.0,104.0,40.7,159.0
35099,female,20,151.0,54.0,19.0,96.0,40.6,96.5
546025,female,20,151.0,54.0,19.0,96.0,40.6,96.5
94628,female,20,152.0,54.0,14.0,96.0,40.2,68.5
...,...,...,...,...,...,...,...,...
594891,male,76,189.0,93.0,29.0,112.0,40.8,257.0
595662,male,78,174.0,77.0,24.0,107.0,40.7,203.0
738755,male,78,174.0,77.0,24.0,107.0,40.7,203.0
725834,male,79,182.0,83.0,26.0,110.0,40.8,240.5


In [14]:
train = train.drop_duplicates(keep='first', ignore_index=True)

In [15]:
train.duplicated().sum()

0

In [16]:
train.shape

(742158, 8)

# Numerical distributions split by categorical variable: Sex

In [17]:
train.groupby('Sex')[TARGET].describe()

,count,mean,std,min,25%,50%,75%,max
Sex,,,,,,,,
female,371664.0,87.565346,57.949142,1.0,37.0,80.0,134.0,300.0
male,370494.0,89.137110,66.618044,1.0,31.0,74.0,138.0,314.0


In [18]:
train.groupby('Sex')[NUM_COLS].describe().T

Sex                      female           male
Age        count  371664.000000  370494.000000
           mean       41.297005      41.583462
           std        15.404135      14.971589
           min        20.000000      20.000000
           25%        28.000000      29.000000
           50%        40.000000      40.000000
           75%        53.000000      52.000000
           max        79.000000      79.000000
Height     count  371664.000000  370494.000000
           mean      165.040225     184.385977
           std         8.219767       8.604026
           min       126.000000     141.000000
           25%       159.000000     179.000000
           50%       164.000000     184.000000
           75%       171.000000     191.000000
           max       222.000000     222.000000
Weight     count  371664.000000  370494.000000
           mean       63.740333      86.582403
           std         7.030128       8.961743
           min        36.000000      46.000000
           25%        59.000000      81.000000
           50%        63.000000      87.000000
           75%        68.000000      93.000000
           max       111.000000     132.000000
Duration   count  371664.000000  370494.000000
           mean       15.503759      15.353577
           std         8.262345       8.449896
           min         1.000000       1.000000
           25%         8.000000       8.000000
           50%        16.000000      15.000000
           75%        23.000000      23.000000
           max        30.000000      30.000000
Heart_Rate count  371664.000000  370494.000000
           mean       95.398763      95.581273
           std         9.417383       9.498284
           min        67.000000      67.000000
           25%        88.000000      88.000000
           50%        96.000000      95.000000
           75%       103.000000     103.000000
           max       128.000000     128.000000
Body_Temp  count  371664.000000  370494.000000
           mean       40.048189      40.024757
           std         0.765013       0.794502
           min        37.100000      37.100000
           25%        39.600000      39.600000
           50%        40.300000      40.200000
           75%        40.600000      40.700000
           max        41.500000      41.500000

# Correlation

In [19]:
train[NUM_COLS].corrwith(train[TARGET])

Age           0.146183
Height       -0.003062
Weight        0.016889
Duration      0.959843
Heart_Rate    0.908528
Body_Temp     0.828680
dtype: float64

In [20]:
train[NUM_COLS].corr(method='pearson')

,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
Age,1.000000,0.011878,0.073569,0.016078,0.017535,0.030623
Height,0.011878,1.000000,0.957751,-0.028914,-0.012302,-0.033202
Weight,0.073569,0.957751,1.000000,-0.019804,-0.001433,-0.022355
Duration,0.016078,-0.028914,-0.019804,1.000000,0.874882,0.903041
Heart_Rate,0.017535,-0.012302,-0.001433,0.874882,1.000000,0.795597
Body_Temp,0.030623,-0.033202,-0.022355,0.903041,0.795597,1.000000


In [21]:
train[NUM_COLS].corr(method='kendall')

,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
Age,1.000000,0.010869,0.051653,0.007973,0.005731,0.017503
Height,0.010869,1.000000,0.842658,-0.020576,-0.010826,-0.018053
Weight,0.051653,0.842658,1.000000,-0.014633,-0.004044,-0.011525
Duration,0.007973,-0.020576,-0.014633,1.000000,0.714370,0.832109
Heart_Rate,0.005731,-0.010826,-0.004044,0.714370,1.000000,0.661639
Body_Temp,0.017503,-0.018053,-0.011525,0.832109,0.661639,1.000000


In [22]:
train[NUM_COLS].corr(method='spearman')

,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
Age,1.000000,0.015774,0.075699,0.011803,0.008513,0.025559
Height,0.015774,1.000000,0.961929,-0.029978,-0.015800,-0.026028
Weight,0.075699,0.961929,1.000000,-0.021149,-0.005815,-0.016411
Duration,0.011803,-0.029978,-0.021149,1.000000,0.884284,0.944945
Heart_Rate,0.008513,-0.015800,-0.005815,0.884284,1.000000,0.840893
Body_Temp,0.025559,-0.026028,-0.016411,0.944945,0.840893,1.000000


In [23]:
reduced_features = [f for f in FEATURES if f not in ('Sex', 'Height', 'Weight')]